# 02 · Validación de calidad
Aplica las reglas de calidad al lote de Bronze y lo separa en:
* **válidos** → `silver.stg_validos` (tipados y normalizados; los consume el notebook 03)
* **rechazados** → `silver.envios_rechazados` (fila original + motivo)
* **métricas** → `silver.calidad_metricas`

Además hay una **puerta de calidad**: si se rechaza más del `max_pct_rechazo` % del lote, la tarea falla y el job se detiene.

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from pyspark.sql import functions as F
from src.config import CATALOGO_POR_DEFECTO, T_BRONZE, T_RECHAZOS, T_METRICAS, T_STG
from src.transformaciones import REGLAS, separar_validos_rechazados
from src.utilidades import escribir_lote

dbutils.widgets.text("catalogo", CATALOGO_POR_DEFECTO)
dbutils.widgets.text("lote", "lote_01")
dbutils.widgets.text("max_pct_rechazo", "50")
catalogo, lote = dbutils.widgets.get("catalogo"), dbutils.widgets.get("lote")
max_pct = float(dbutils.widgets.get("max_pct_rechazo"))

bronze = spark.table(f"{catalogo}.{T_BRONZE}").filter(F.col("lote_id") == lote)

## 1. Reglas de calidad documentadas
| Regla | Condición que debe cumplirse |
|---|---|
| R01 | `order_id` y `order_item_id` no nulos |
| R02 | `customer_id` no nulo |
| R03 | fecha de pedido interpretable |
| R04 | `sales` y `order_item_total` válidos y ≥ 0 |
| R05 | cantidad > 0 |
| R06 | fecha de envío ≥ fecha de pedido |

In [0]:
validos, rechazados, por_regla, filas_entrada = separar_validos_rechazados(bronze)
n_rechazadas = rechazados.count()
n_validas = filas_entrada - n_rechazadas
print(f"Entrada: {filas_entrada} | Válidas: {n_validas} | Rechazadas: {n_rechazadas}")
for regla, n in por_regla.items():
    print(f"  {regla:<35} incumplen: {n}")

## 2. Guardar rechazos, métricas y válidos

In [0]:
escribir_lote(spark, rechazados, f"{catalogo}.{T_RECHAZOS}", lote)

metricas = [("filas_entrada", filas_entrada), ("filas_validas", n_validas), ("filas_rechazadas", n_rechazadas)]
metricas += [(f"incumplen:{r}", n) for r, n in por_regla.items()]
df_m = (spark.createDataFrame([(lote, m, int(v)) for m, v in metricas], "lote_id string, metrica string, valor long")
        .withColumn("fecha_proceso", F.current_timestamp()))
escribir_lote(spark, df_m, f"{catalogo}.{T_METRICAS}", lote)

# staging del lote (se sobrescribe en cada ejecución; solo contiene el lote actual)
(validos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalogo}.{T_STG}"))
print(f"Staging escrito: {spark.table(f'{catalogo}.{T_STG}').count()} filas")

display(spark.table(f"{catalogo}.{T_RECHAZOS}").filter(F.col("lote_id") == lote)
        .select("order_id", "order_item_id", "customer_id", "sales", "motivo_rechazo").limit(30))

## 3. Puerta de calidad

In [0]:
pct = 100.0 * n_rechazadas / filas_entrada if filas_entrada else 0.0
print(f"Rechazo del lote: {pct:.2f} % (máximo permitido: {max_pct} %)")
if pct > max_pct:
    raise Exception(f"Calidad insuficiente: {pct:.2f} % de filas rechazadas (> {max_pct} %). Lote detenido.")